# Mini devoir 6 : Itérations QR et valeurs propres

La décomposition QR est une factorisation utile d'une matrice, largement utilisée en algèbre linéaire numérique. Elle consiste à écrire une matrice $A\in\mathbb{R}^{n\times n}$ sous la forme :

$$A = QR,$$

où les colonnes de $Q$ sont orthonormales et $R$ est triangulaire supérieure. Dans ce devoir, nous utiliserons la décomposition QR pour trouver les états propres et les niveaux d'énergie de l'oscillateur quantique anharmonique :

$$H = \frac{p^2}{2m}+\frac{1}{2}m\omega^2x^2 + \sum_{k=3}^N c_kx^k$$

En général, ce système ne possède pas de solution sous forme fermée ; il doit donc être résolu numériquement.


# 1. Mise en place du problème [6 points]

**Prenez le Hamiltonien du système et effectuez un changement de variables en $x$ afin que l'équation de Schrödinger indépendante du temps puisse s'écrire :**

$$-\frac{d^2\psi}{dx^2} + \left(x^2+\sum_{k=3}^Nc_kx^k\right)\psi=E\psi$$

(Les quantités $x$, $c$ et $E$ sont les variables que vous redéfinirez.)

L'équation de Schrödinger est:

$$H\psi = \epsilon\psi$$
où $\epsilon$ est l'énergie.
L'opérateur quantité de mouvement (p) vaut $-iℏ\frac{d}{dX}$ où X est la position.

En remplaçant H dans l'équation de Schrodinger, on a:


$$\left(\frac{-ℏ^2}{2m}\frac{d^2}{dX^2} + \frac{1}{2}m\omega^2X^2 + \sum_{k=3}^Na_kX^k\right)\psi=\epsilon\psi$$

La longueur caractéristique de l’oscillateur harmonique est 
$$ l = \sqrt{\frac{\hbar}{m\omega}}$$
Si on fait le changement de variable $x =\frac{X}{l}$, on trouve

$$\frac{-\hbar\omega}{2}\frac{d^2\psi}{dx^2} + \left(\frac{\hbar \omega}{2}x^2 + \sum_{k=3}^Na_k\left(\frac{\hbar}{m\omega}\right)^{k/2}x^k\right)\psi=\epsilon\psi$$

Si on pose que $E = \frac{2\epsilon}{\hbar\omega}$ et qu'on fait le changement de variable suivant, $ a_k\frac{2}{\hbar\omega}\left(\frac{\hbar}{m\omega}\right)^{k/2} = c_k$

$$-\frac{d^2\psi}{dx^2} + \left(x^2+\sum_{k=3}^Nc_kx^k\right)\psi=E\psi$$

In [1]:
import numpy as np
from scipy.sparse import diags
from numpy.linalg import norm, lstsq, cond
import matplotlib.pyplot as plt

# construire la systeme
def systeme(n, cs):
    # maillage
    xs = np.linspace(-1, 1, n)
    dx = xs[1] - xs[0]

    # energy cinetique
    diag_1 = -2 * np.ones(n) / dx
    diag_2 = 1 * np.ones(n-1) / dx
    D2 = diags([diag_2, diag_1, diag_2], offsets=[-1, 0, 1], format='csr')

    # energie potentiel
    def V(x):
        result = x**2
        for (k, c) in enumerate(cs):
            result += c * x**(k+3)
        return result
    V = diags([[V(x) for x in xs]], offsets=[0], format='csr')

    H = -D2 + V
    return xs, H.toarray()

# 2. Décomposition QR en utilisant des rotations de Givens

Afin d'effectuer concrètement la décomposition QR, nous commencerons par prendre $Q=I$ et $R = H$. Nous effectuerons ensuite itérativement des rotations sur $Q$ et $R$ jusqu'à ce que $R$ soit triangulaire. Par exemple, supposons que $R$ soit une matrice $3\times3$ :

$$\begin{bmatrix}*&*&*\\a&b&*\\c&d&*\end{bmatrix}$$

Dans ce cas, nous devons d'abord effectuer une rotation qui ammène $c$ à zéro ; cela s'appelle une rotation de Givens. Une fois cette rotation $G_0$ identifiée, on insère $G_0^TG_0$ de sorte que :

$$\begin{align}H &= H\\ &= IG_0^TG_0H\end{align}$$

Cela signifie que nous avons défini la première itération QR comme $Q_0 = G_0^T$, $R_0 = G_0H$. En poursuivant les itérations pour éliminer davantage d'éléments de $R$, nous insérons davantage de termes $G^TG$. On obtient alors :

$$H= G_0^TG_1^TG_2^T\cdots G_2G_1G_0H$$

En pratique, nous ne formons pas explicitement ces matrices, mais plutôt nous modifions la matrice.

**Modifiez la section « TODO » pour implémenter les rotations dans la décomposition QR en utilisant le cos et le sin obtenus à partir de la fonction `rotation`.**


In [ ]:
# rotation Givens
def rotation(a, b):
    return (a, -b) / np.sqrt(a**2 + b**2)

# decomposition QR
def qr(A):
    m, n = A.shape
    R = A.copy()
    Q = np.identity(m)
    for i in range(0, n - 1):
        for j in range(i + 1, m):
            cos, sin = rotation(R[i, i], R[j, i])
            # TODO 2. utiliser les rotation de Givens pour obtenir Q et R
            if R[j, i] != 0:
                R[i], R[j] = R[i], R[j]
                Q[:, i], Q[:, j] = Q[:, i], Q[:, j]

                 # ---- appliquer rotation sur R (lignes i et j) ----
                Ri = R[i, :].copy()
                Rj = R[j, :].copy()

                R[i, :] = cos * Ri - sin * Rj
                R[j, :] = sin * Ri + cos * Rj

                # ---- appliquer rotation sur Q (colonnes i et j) ----
                Qi = Q[:, i].copy()
                Qj = Q[:, j].copy()

                Q[:, i] = cos * Qi - sin * Qj
                Q[:, j] = sin * Qi + cos * Qj

            #############################################################
    return Q, R


# --------- TEST ---------
#A = np.array([[1, 2, 3],
#              [4, 5, 6],
#              [7, 8, 9]], dtype=float)

#Q, R = qr(A)

#print(Q)
#print(R)
#print(Q @ R)
#print(Q @ np.transpose(Q))

[[ 0.12309149  0.90453403 -0.40824829]
 [ 0.49236596  0.30151134  0.81649658]
 [ 0.86164044 -0.30151134 -0.40824829]]
[[8.12403840e+00 9.60113630e+00 1.10782342e+01]
 [2.63828304e-16 9.04534034e-01 1.80906807e+00]
 [3.57225212e-16 5.55111512e-17 0.00000000e+00]]
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
[[ 1.00000000e+00 -2.44930592e-17 -3.30781356e-17]
 [-2.44930592e-17  1.00000000e+00 -6.06602824e-17]
 [-3.30781356e-17 -6.06602824e-17  1.00000000e+00]]


# 3. Taux de convergence de l'*itération* QR [6 points]

L'*itération* QR est une méthode qui utilise la *décomposition* QR pour trouver les valeurs propres et les vecteurs propres. Pour une matrice $H$, on commence avec $H_0 = Q_0R_0$ :

1. Créer $H_{k}=R_{k-1}Q_{k-1}$ (N.B. que c'est RQ et pas QR)
2. Former la décomposition $H_{k+1} = Q_kR_k$
3. Revenir à l'étape 1.

Ce procédé construit :

$$H_k = Q^T_{k-1}Q^T_{k-2}Q^T_{k-3}\cdots Q_0HQ_0\cdots Q_{k-3}Q_{k-2}Q_{k-1}$$

On peut démontrer que ce produit de matrices $Q$ converge vers les vecteurs propres de $H$. Cela signifie que la diagonale de $H_k$ converge vers les valeurs propres de $H$. Ceci a été implémenté ci-dessous pour l'oscillateur anharmonique avec des conditions de frontière Dirichlet.

**Tracez la différence par itération des vecteurs propres en fonction de la taille de pas et du nombre de conditionnement. Quels sont les ordres de convergence ?**


In [ ]:
# iteration QR
def eig_qr(A, tol=1e-5):
    Q, R = qr(A)
    vs = Q
    T = np.dot(R, Q)
    l = T[0, 0]
    diff = 1
    i = 0
    while diff > tol:
        i += 1
        Q, R = qr(T)
        vs = np.dot(vs, Q)
        T = np.dot(R, Q)
        diff = abs(l - T[0, 0]) / abs(l)
        l = T[0, 0]
    ls = np.array([T[i, i] for i in range(len(T))])
    return ls, vs, i


ns = np.pow(2, range(3, 8))

#TODO les ordres de convergence

###############################

# 4. Propagation et décomposition [6 points]

Jusqu'à présent, nous avons déterminé les modes propres du système. Nous allons maintenant écrire le code permettant de décrire l'évolution temporelle du système. En général, nous n'avons pas besoin de tous les modes pour représenter correctement une fonction d'onde. Cela permet d'effectuer moins d'opérations puisqu'on travaille dans une base plus petite.

**Modifiez la section « TODO » pour décomposer $\phi$ sur les modes propres, le faire évoluer dans le temps, puis revenir dans la base originale. Utilisez l'argument `k`, qui détermine combien de modes propres sont utilisés pour la décomposition.**


In [25]:
def propag(ls, vs, phi, t, k=5):
    # TODO 4. decomposer phi et propager en temps

    return phi
    #############################################

n = 64
xs, H = systeme(n, [1, 2, 3])
ls, vs, it = eig_qr(H)

mu = 0.5
sigma = 1e-1
phi = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-(xs-mu)**2/(2*sigma**2))
phi /= norm(phi)

# 5. Créer une animation de la simulation [2 points]

Ce problème vise davantage une démonstration visuelle qu'une évaluation. Nous allons ici créer un gif de l'évolution temporelle de l'oscillateur anharmonique.

**Modifiez la section « TODO » pour déterminer le nombre de modes ($K$) nécessaire afin d'approximer $\phi$ de la partie 4 (précision $\sqrt{\varepsilon}$).**

Vous pouvez également vérifier visuellement comment la qualité de l'approximation change lorsque vous utilisez de moins en moins d'information.


In [ ]:
err = np.inf
K = 1
#TODO déterminer K

##################


from matplotlib.animation import FuncAnimation
fig, ax = plt.subplots()
line, = ax.plot(xs, phi)
ax.set_ylim(0.0, 0.2)

def update(t):
    line.set_ydata(np.abs(propag(ls, vs, phi, t, k=K))**2)
    return line,

anim = FuncAnimation(fig, update,
                     frames=np.linspace(0, 7, 1000),
                     interval=30,
                     blit=True)
anim.save("animation.gif", writer="pillow", fps=30)